# Витрина транзакций ТЭ — 7. Два первых пункта анализа

AI-14135. Пункты 1 и 2: сравнение распределений и косинус портретов. Isolation Forest и кластеризация здесь не считаются.

Вход — уже собранный профиль `sandbox_ai.tmp_shestopalov_te_profile`, одна строка на ИНН. Операции заново не читаются. p10 и p90 чека в профиле нет, тесты идут по остальным полям портрета.

Группа меньше 30 клиентов в сравнение распределений и в косинус не входит. Клиенты без кода точки (`mcc_code = не заполнен`) убираются до расчётов: это пустой код, не вид деятельности.

Тетрадка импортирует `te_trx_analysis.py` из той же папки. Если импорт не находит файл, откройте тетрадку из каталога `te_trx_mart`.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
import sys
from pathlib import Path

import pandas as pd

found = None
for cand in [Path.cwd(), *Path.cwd().parents]:
    if (cand / "te_trx_analysis.py").exists():
        found = cand
        break
    if (cand / "te_trx_mart" / "te_trx_analysis.py").exists():
        found = cand / "te_trx_mart"
        break
if found is None:
    raise FileNotFoundError(
        "Не найден te_trx_analysis.py. Откройте тетрадку из папки te_trx_mart."
    )
sys.path.insert(0, str(found))
import te_trx_analysis as analysis

with imp:
    profile = imp.fetch("select * from sandbox_ai.tmp_shestopalov_te_profile")

before = len(profile)
profile = profile.loc[profile["mcc_code"].astype(str) != "не заполнен"].copy()
print(f"Клиентов в профиле: {before}. Без кода точки убрано: {before - len(profile)}. В расчёте: {len(profile)}.")
profile.head()


## Численность групп

`достаточно` — от 100 клиентов, `на грани` — от 30 до 99, `мало` — меньше 30. Тесты распределений и косинус считаются только там, где клиентов не меньше 30, и в дополнении группы тоже не меньше 30.


In [ ]:
coverage = analysis.coverage_table(profile)
summary = (
    coverage.groupby(["slice_name", "enough"], as_index=False)
    .agg(groups=("slice_value", "count"), clients=("client_cnt", "sum"))
    .sort_values(["slice_name", "enough"])
)
display(summary)
usable = coverage.loc[coverage["enough"] != "мало"].copy()
print(f"Групп, где клиентов не меньше 30: {len(usable)}.")
display(usable)


## 1. Сравнение распределений

Каждая группа сравнивается с остальными клиентами того же разреза. Колмогоров–Смирнов и Манн–Уитни показывают, различаются ли распределения. Wasserstein — насколько далеко сдвинуты суммы и количества. Jensen–Shannon — насколько различается форма. p-значения Колмогорова–Смирнова и Манна–Уитни поправлены по Бенджамини–Хохбергу внутри разреза и метрики.

Ниже сначала строки, где после поправки p < 0.05, с самым большим сдвигом Wasserstein. Затем хи-квадрат и V Крамера: связаны ли группа MCC и регион, группа MCC и размер бизнеса, класс ОКВЭД и регион, класс ОКВЭД и размер. Это связь категорий, не сравнение сумм чека.


In [ ]:
tests = analysis.compare_distributions(profile)
if tests.empty:
    print("Нет группы, где и она сама, и остальные имеют не меньше 30 клиентов.")
else:
    strong = tests.loc[tests["ks_pvalue_bh"] < 0.05].sort_values("wasserstein", ascending=False)
    print(f"Сравнений: {len(tests)}. После поправки p < 0.05: {len(strong)}.")
    cols = ["slice_name", "slice_value", "metric", "n_group", "n_rest", "wasserstein", "jensen_shannon", "ks_pvalue_bh", "mannwhitney_pvalue_bh"]
    display(strong[cols].head(30))

assoc = analysis.association_tests(profile)
if assoc.empty:
    print("Хи-квадрат не посчитан: нет двух категорий с достаточным числом клиентов.")
else:
    display(assoc)


## 2. Нормализация и косинус

Поля портрета приводятся к одному масштабу (z-нормализация по всем клиентам расчёта). Для каждой группы берётся средний портрет. Косинус близок к 1, когда средние портреты похожи, и к 0, когда они расходятся. Сравниваются группы, а не отдельные клиенты. В пару попадают только группы от 30 клиентов.


In [ ]:
cosine = analysis.cosine_by_slice(profile)
if cosine.empty:
    print("Косинус не посчитан: в разрезе нет двух групп от 30 клиентов.")
else:
    pieces = []
    for slice_name, part in cosine.groupby("slice_name"):
        ordered = part.sort_values("cosine_similarity")
        far = ordered.head(5).assign(pair="далёкие")
        near = ordered.tail(5).assign(pair="близкие")
        pieces.append(pd.concat([far, near], ignore_index=True))
    show = pd.concat(pieces, ignore_index=True)
    display(show[["slice_name", "pair", "group_a", "group_b", "n_a", "n_b", "cosine_similarity"]])
